# 03 — Signal FFT

**Workload:** FFT low-pass filtering and overlapping-window spectral analysis.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import sys

# rnp is the Rust engine's numpy-compatible package: one import swap and
# everything below is ordinary NumPy code.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "shim"))

import rnp as np

probe = np.array(0)
print("rnp version:", np.__version__)
print(f"engine: {type(probe).__module__}.{type(probe).__name__}")
assert type(probe).__module__ == "_rnp"

rnp version: 2.5.2
engine: _rnp.ndarray


## Create a deterministic noisy signal

Combine a 5 Hz sine wave with seeded Gaussian noise.

In [2]:
sample_rate = 64.0
count = 128
time = np.arange(count) / sample_rate
clean = np.sin(2.0 * np.pi * 5.0 * time)
noise = np.random.default_rng(20260825).normal(0.0, 0.35, count)
noisy = clean + noise
print("sample count:", noisy.size)
print("noisy head:", np.round(noisy[:8], 6))

sample count: 128
noisy head: [-0.011595  1.080874  1.219431  0.839927 -0.235048  0.730703  0.311301
 -0.372264]


## Low-pass filter in the frequency domain

Zero bins above 8 Hz, then transform back to the time domain.

In [3]:
frequencies = np.fft.rfftfreq(count, d=1.0 / sample_rate)
spectrum = np.fft.rfft(noisy)
filtered = np.fft.irfft(np.where(frequencies <= 8.0, spectrum, 0.0), n=count)
print("filtered head:", np.round(filtered[:8], 6))

filtered head: [ 0.141536  0.631473  0.931805  0.979178  0.777941  0.391619 -0.079564
 -0.526164]


## Build overlapping spectra

Use stride-trick windows and a Hann taper before measuring spectral energy.

In [4]:
windows = np.lib.stride_tricks.sliding_window_view(noisy, 32)[::16]
tapered = windows * np.hanning(32)
spectrogram = np.abs(np.fft.rfft(tapered, axis=1)) ** 2
peak_bins = np.argmax(spectrogram[:, 1:], axis=1) + 1
window_frequencies = np.fft.rfftfreq(32, d=1.0 / sample_rate)
peak_frequencies = window_frequencies[peak_bins]
spectrogram_energy = spectrogram[:, 1:].sum(axis=1)
print("window peak frequencies:", peak_frequencies)
print("window energies:", np.round(spectrogram_energy, 3))

window peak frequencies: [4. 6. 4. 4. 6. 4. 6.]
window energies: [101.124  98.322 144.297  95.474 123.886 127.464 144.664]


## Verify the result

In [5]:
assert np.array_equal(peak_frequencies, [4.0, 6.0, 4.0, 4.0, 6.0, 4.0, 6.0])
assert np.allclose(filtered[:3], [0.14153594, 0.63147300, 0.93180457], rtol=0.0, atol=1e-8)
print("PASS — all FFT assertions passed.")

PASS — all FFT assertions passed.
